In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data_utils
from dataclasses import dataclass, field
import random
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [2]:
seed = 42

random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [3]:
path = "datasets/winequality.csv"
df = pd.read_csv(path)
df.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,white,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,white,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,white,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [4]:
df = df.dropna()
np.count_nonzero(df.isnull().values)

0

In [5]:
df["type"] = df["type"].map(lambda x: 1 if x =="red" else 0)
df["type"].value_counts()

type
0    4870
1    1593
Name: count, dtype: int64

In [6]:
X = df.drop(columns=["quality"]).to_numpy()
y = df["quality"].to_numpy()

In [7]:
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y,
                                                            test_size = 0.1, random_state=seed,
                                                            stratify=y)

In [8]:
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval,
                                                            test_size = 0.1, random_state=seed,
                                                            stratify=y_trainval)

In [9]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((5234, 12), (5234,), (582, 12), (582,), (647, 12), (647,))

In [10]:
ss = StandardScaler()

X_train = ss.fit_transform(X_train)
X_val = ss.transform(X_val)
X_test = ss.transform(X_test)

In [11]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [12]:
train_df = data_utils.TensorDataset(X_train_tensor, y_train_tensor)
val_df = data_utils.TensorDataset(X_val_tensor, y_val_tensor)
test_df = data_utils.TensorDataset(X_test_tensor, y_test_tensor)

In [13]:
train_dl = data_utils.DataLoader(train_df, batch_size=64, shuffle=True)
val_dl = data_utils.DataLoader(val_df, batch_size=32, shuffle=False)
test_dl = data_utils.DataLoader(test_df, batch_size=32, shuffle=False)

# Definizione classi

In [14]:
class BaselineModel(nn.Module):

  def __init__(self):
    super().__init__()

    self.layer_1 = nn.Linear(in_features=12, out_features=256)
    self.layer_2 = nn.Linear(in_features=256, out_features=128)
    self.layer_3 = nn.Linear(in_features=128, out_features=64)
    self.layer_4 = nn.Linear(in_features=64, out_features=1) #Regression task

  def forward(self,x): # [256, 12]
    x = F.relu(self.layer_1(x))
    x = F.relu(self.layer_2(x))
    x = F.relu(self.layer_3(x))
    x = self.layer_4(x) #[256, 1]
    return x
  
val_loss_values_baseline = []
train_loss_values_baseline = []

In [15]:
baseline = BaselineModel()

In [16]:
class EarlyStopping:
    def __init__(self, save_path, patience=5, min_delta=0):

        self.save_path = save_path
        self.patience = patience
        self.min_delta = min_delta
        self.min_val_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, validation_loss, model):

        if self.min_val_loss is None:     #Prima epoca
          self.min_val_loss = validation_loss
          self.save_checkpoint(model)

        elif (self.min_val_loss - validation_loss) > self.min_delta: #Epoca con miglioramento
          self.min_val_loss = validation_loss
          self.save_checkpoint(model)
          self.counter = 0


        else:     #Nessun miglioramento
          self.counter +=1
          if self.counter >= self.patience:
            self.early_stop = True

    def save_checkpoint(self, model):
      torch.save(model.state_dict(), self.save_path)

In [17]:
@dataclass
class Experiment:
  
  name: str
  checkpoints_folder: str
  checkpoint_name:str
  model: object
  use_early_stopping: bool
  loss_fn : object
  optimizer: object
  color: str
  alpha: float
  val_mse: float = None
  lr: float = 1e-5
  epochs: int = 600
  patience: int = 10
  min_delta: float = 0
  epoch_count: list = field(default_factory=list)
  val_loss_values: list = field(default_factory=list)
  train_loss_values: list = field(default_factory=list)
  plt_args_training: dict = field(default_factory=dict)
  plt_args_validation: dict = field(default_factory=dict)


  def __post_init__(self):
    self.optimizer = self.optimizer(params=self.model.parameters(),
                            lr=self.lr)
    self.checkpoints_folder = os.path.join(self.checkpoints_folder, self.name)
    os.makedirs(self.checkpoints_folder, exist_ok = True)
    self.checkpoint_save_path = os.path.join(self.checkpoints_folder, self.checkpoint_name)

In [18]:
def fit(exp:Experiment, trainloader, valloader):
  exp.epoch_count = []
  exp.train_loss_values = []
  exp.val_loss_values = []
  if exp.use_early_stopping:
    early_stopping = EarlyStopping(exp.checkpoint_save_path, patience=exp.patience, min_delta=exp.min_delta)
  for epoch in range(exp.epochs):
    exp.model.train()
    loss_epoch = 0
    for i, data in enumerate(trainloader, 0):
      X = data[0]
      y = data[1]
      y_pred = exp.model(X)
      loss = exp.loss_fn(y_pred.squeeze(-1), y)

      loss_epoch += loss

      exp.optimizer.zero_grad()

      loss.backward()

      exp.optimizer.step()
    loss_val = 0
    exp.model.eval()
    for j, data in enumerate(valloader, 0):

      X = data[0]
      y = data[1]

      with torch.no_grad():

        y_pred = exp.model(X)
        loss = exp.loss_fn(y_pred.squeeze(-1), y)

      loss_val += loss
    exp.epoch_count.append(epoch)
    exp.train_loss_values.append(loss_epoch.detach().numpy()/len(trainloader))
    exp.val_loss_values.append(loss_val.detach().numpy()/len(valloader))


    print(f"Epoca: {epoch} |  Train Loss: {loss_epoch/len(trainloader)} | Val Loss: {loss_val/len(valloader)} ")

    if exp.use_early_stopping:
      early_stopping(loss_val/len(valloader), exp.model)
      if early_stopping.early_stop:
        print("Early stopping all'epoca:", epoch)
        break
      exp.model.load_state_dict(torch.load(exp.checkpoint_save_path))

In [19]:
def evaluate(exp:Experiment, testloader):
  tot_loss = 0
  exp.model.eval()
  for j, data in enumerate(testloader, 0):

    X = data[0]
    y = data[1]

    with torch.no_grad():

      y_pred = exp.model(X)
      loss = exp.loss_fn(y_pred, y)

    tot_loss += loss
  return tot_loss.detach().numpy()/len(testloader)


In [20]:
experiments = []

In [21]:
baseline_exp = Experiment(
  name = "baseline",
  checkpoints_folder = 'models',
  checkpoint_name = "model.pt",
  model = baseline,
  use_early_stopping = False,
  loss_fn = nn.MSELoss(),
  optimizer= torch.optim.Adam,
  lr = 1e-3,
  epochs = 100,
  epoch_count =  100,
  val_loss_values = val_loss_values_baseline,
  train_loss_values = train_loss_values_baseline,
  color = "#FF7F0E",
  alpha = .3,
)
experiments.append(baseline_exp)

In [22]:
torch.manual_seed(seed)
# Modello con Early stopping
early_stopping_model = BaselineModel()
# Loss function
loss_fn = nn.MSELoss() # siamo in un problema di regressione

# Adam optimizer
optimizer = torch.optim.Adam(params=early_stopping_model.parameters(),
                            lr=1e-3)

epochs = 1000
train_loss_values_early_stopping = []
val_loss_values_early_stopping = []
epoch_count_early_stopping = []

In [23]:
early_stopping_exp = Experiment(
  name = "early_stopping",
  checkpoints_folder = 'models',
  checkpoint_name = "model.pt",
  model = early_stopping_model,
  use_early_stopping = True,
  loss_fn = nn.MSELoss(),
  optimizer= torch.optim.Adam,
  lr = 1e-4,
  epochs = 100,
  patience = 50,
  epoch_count =  epoch_count_early_stopping,
  val_loss_values = val_loss_values_early_stopping,
  train_loss_values =  train_loss_values_early_stopping,
  color = "#4D61E2",
  alpha = .8
)

experiments.append(early_stopping_exp)

In [24]:
experiments

[Experiment(name='baseline', checkpoints_folder='models\\baseline', checkpoint_name='model.pt', model=BaselineModel(
   (layer_1): Linear(in_features=12, out_features=256, bias=True)
   (layer_2): Linear(in_features=256, out_features=128, bias=True)
   (layer_3): Linear(in_features=128, out_features=64, bias=True)
   (layer_4): Linear(in_features=64, out_features=1, bias=True)
 ), use_early_stopping=False, loss_fn=MSELoss(), optimizer=Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     decoupled_weight_decay: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.001
     maximize: False
     weight_decay: 0
 ), color='#FF7F0E', alpha=0.3, val_mse=None, lr=0.001, epochs=100, patience=10, min_delta=0, epoch_count=100, val_loss_values=[], train_loss_values=[], plt_args_training={}, plt_args_validation={}),
 Experiment(name='early_stopping', checkpoints_folder='models\\early_stopping', checkpoint_name=

In [25]:
# Kfold cross-validation

kf = KFold(n_splits=3, shuffle=True, random_state=seed)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_tensor)):

    # Creating Tensors for this fold
    X_train_fold, X_val_fold = X_train_tensor[train_idx], X_train_tensor[val_idx]
    y_train_fold, y_val_fold = y_train_tensor[train_idx], y_train_tensor[val_idx]

    # Creating DataSets and DataLoaders for this fold
    train_df_split = data_utils.TensorDataset(X_train_fold, y_train_fold)
    val_df_split = data_utils.TensorDataset(X_val_fold, y_val_fold)

    train_dl_split = data_utils.DataLoader(train_df, batch_size=64, shuffle=True)
    val_dl_split = data_utils.DataLoader(val_df, batch_size=32, shuffle=False)

    # Training models for this fold
    for exp in experiments:
        fit(exp, train_dl_split, val_dl_split)

        # Evaluate the model on the test set
        test_loss = evaluate(exp, test_dl)
        print(f"Test Loss for {exp.name} on fold {fold}: {test_loss}")


Epoca: 0 |  Train Loss: 7.2000837326049805 | Val Loss: 1.3502000570297241 
Epoca: 1 |  Train Loss: 1.20016610622406 | Val Loss: 0.8426365256309509 
Epoca: 2 |  Train Loss: 0.7193638682365417 | Val Loss: 0.6303158402442932 
Epoca: 3 |  Train Loss: 0.5429238677024841 | Val Loss: 0.5811581611633301 
Epoca: 4 |  Train Loss: 0.5013422966003418 | Val Loss: 0.5537623763084412 
Epoca: 5 |  Train Loss: 0.4880193769931793 | Val Loss: 0.570544421672821 
Epoca: 6 |  Train Loss: 0.4851214289665222 | Val Loss: 0.533888578414917 
Epoca: 7 |  Train Loss: 0.46953079104423523 | Val Loss: 0.5397956371307373 
Epoca: 8 |  Train Loss: 0.4677894413471222 | Val Loss: 0.5517632961273193 
Epoca: 9 |  Train Loss: 0.4557415246963501 | Val Loss: 0.5529084205627441 
Epoca: 10 |  Train Loss: 0.45667123794555664 | Val Loss: 0.5476681590080261 
Epoca: 11 |  Train Loss: 0.45016545057296753 | Val Loss: 0.556236982345581 
Epoca: 12 |  Train Loss: 0.45377016067504883 | Val Loss: 0.5420066118240356 
Epoca: 13 |  Train Loss

c:\Repositories\proai\course6-pytorch\.venv\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
c:\Repositories\proai\course6-pytorch\.venv\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([7])) that is different to the input size (torch.Size([7, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoca: 0 |  Train Loss: 28.923128128051758 | Val Loss: 18.911710739135742 
Epoca: 1 |  Train Loss: 8.101222038269043 | Val Loss: 2.5576584339141846 
Epoca: 2 |  Train Loss: 2.417147636413574 | Val Loss: 1.7927888631820679 
Epoca: 3 |  Train Loss: 1.9407105445861816 | Val Loss: 1.533286452293396 
Epoca: 4 |  Train Loss: 1.6804090738296509 | Val Loss: 1.3606534004211426 
Epoca: 5 |  Train Loss: 1.4859049320220947 | Val Loss: 1.2225816249847412 
Epoca: 6 |  Train Loss: 1.3207937479019165 | Val Loss: 1.1082994937896729 
Epoca: 7 |  Train Loss: 1.1765252351760864 | Val Loss: 1.000780701637268 
Epoca: 8 |  Train Loss: 1.0420184135437012 | Val Loss: 0.9140486717224121 
Epoca: 9 |  Train Loss: 0.9266335368156433 | Val Loss: 0.8268642425537109 
Epoca: 10 |  Train Loss: 0.8226414322853088 | Val Loss: 0.7599214315414429 
Epoca: 11 |  Train Loss: 0.7402897477149963 | Val Loss: 0.7074862718582153 
Epoca: 12 |  Train Loss: 0.6700338125228882 | Val Loss: 0.6728217601776123 
Epoca: 13 |  Train Loss: 0